# Sequences & RNN — Unfolding Time

An RNN processes one time step at a time, carrying **hidden state** forward.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
torch.manual_seed(42); np.random.seed(42)


## Synthetic sequence data

In [ ]:
class DummyDataGenerator:
    def __init__(self, seq_len=20, n=1):
        t = torch.linspace(0, 4*np.pi, seq_len)
        self.data = torch.sin(t) + 0.1*torch.randn(seq_len)
        self.seq_len = seq_len
    def get_series(self): return self.data

class SequenceDataset(Dataset):
    def __init__(self, data, window=5):
        self.data = data; self.window = window
    def __len__(self): return len(self.data) - self.window
    def __getitem__(self, i):
        x = self.data[i:i+self.window]
        y = self.data[i+self.window]
        return x.unsqueeze(-1), y

gen = DummyDataGenerator()
data = gen.get_series()
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(data.numpy(), 'b-', lw=2); ax.set_title('Time series — each point is one time step')
ax.set_xlabel('time t'); ax.set_ylabel('value')
plt.show()

## RNN cell — hidden state at each step

In [ ]:
class TinyRNN(nn.Module):
    def __init__(self, d_in=1, hidden=8):
        super().__init__()
        self.rnn = nn.RNN(d_in, hidden, batch_first=True)
    def forward(self, x):
        out, h = self.rnn(x)
        return out, h

x = data[:10].unsqueeze(0).unsqueeze(-1)  # (1, 10, 1)
model = TinyRNN()
out, h = model(x)
print(f"output per step shape: {out.shape}, final hidden: {h.shape}")

fig, axes = plt.subplots(2, 1, figsize=(10, 5))
axes[0].plot(out.squeeze().detach().numpy(), 'g-o', label='RNN output each step')
axes[0].plot(x.squeeze().numpy(), 'b--', alpha=0.5, label='input')
axes[0].legend(); axes[0].set_title('RNN outputs at each time step')
axes[1].bar(range(out.shape[-1]), h.squeeze().detach().numpy(), color='coral')
axes[1].set_title('Final hidden state vector (memory)')
plt.tight_layout(); plt.show()